<a href="https://colab.research.google.com/github/briliananugra/mobile-legends-sentiment-analysis/blob/main/mobile_legends_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/briliananugra/mobile-legends-sentiment-analysis/main/data/ml_sample_raw.csv"
df_sample = pd.read_csv(url)

print(df_sample.shape)
df_sample.head()

(3000, 3)


,content,score,at
0,woy monton bisa buat game gak udah empat belas...,1,2026-07-27 07:57:30
1,KALAU NGASIH TEAM YG BENER LAHHH,1,2026-07-27 07:54:47
2,mooton bujang intinya game pp game bujang main...,1,2026-07-27 07:53:01
3,capek juga main selalu kalah saya sebagai solo...,1,2026-07-27 07:52:40
4,"Pengen balik ke ml tolong, jangan tambahkan fi...",2,2026-07-27 07:52:24


---
## 📌 Catatan: Sel Eksplorasi Awal (One-Time)

Sel-sel di bawah ini (distribusi score, cek data kosong/duplikat, contoh teks, dan simpan CSV) adalah **eksplorasi awal Checkpoint 1** yang sudah dilakukan sekali dan hasilnya sudah didokumentasikan.

- **Tidak perlu dijalankan ulang** setiap buka notebook — cukup dijalankan kalau ingin verifikasi ulang data.
- Data hasil eksplorasi ini sudah tersimpan permanen di GitHub: `data/ml_sample_raw.csv`
- Sel ini dipertahankan sebagai dokumentasi proses penelitian (bukti kerja analitis), bukan langkah wajib di setiap sesi.
---

In [ ]:
# distribusi rating
print("Distribusi score:")
print(df_sample['score'].value_counts().sort_index())

# cek kosong & duplikat
print("\nData kosong per kolom:")
print(df_sample.isnull().sum())
print("\nJumlah duplikat (content):", df_sample['content'].duplicated().sum())

# contoh teks untuk cek kualitas (bahasa gaul, emoji, dll)
print("\nContoh 5 ulasan:")
for i, t in enumerate(df_sample['content'].sample(5, random_state=42)):
    print(f"{i+1}. {t}\n")

Distribusi score:
score
1    1695
2     170
3     136
4     141
5     858
Name: count, dtype: int64

Data kosong per kolom:
content    0
score      0
at         0
dtype: int64

Jumlah duplikat (content): 154

Contoh 5 ulasan:
1. setiap proses pertandingan kalo tim musuh atau tim sendiri pake skin animasi ga bisa masuk loading nya lama udah gitu dihitung afk

2. mantap

3. game nya gak enak , di kasih tim bot semua di kasih kalah terus naik kagak yang ada malah buang buang waktu aja

4. keren

5. menurut ku game ini bagus tapi tambah bnyak skin lagi bagus sihh



> ⚠️ Sel ini tidak perlu dijalankan ulang — data sudah tersimpan di GitHub (`data/ml_sample_raw.csv`) dan tidak berubah.

In [ ]:
df_sample.to_csv('/content/ml_sample_raw.csv', index=False)
print("Tersimpan: /content/ml_sample_raw.csv")

Tersimpan: /content/ml_sample_raw.csv


# Tahap Preprocessing

In [ ]:
!pip install -q Sastrawi emoji

import pandas as pd

url = "https://raw.githubusercontent.com/briliananugra/mobile-legends-sentiment-analysis/main/data/ml_sample_raw.csv"
df = pd.read_csv(url)
print("Sebelum drop duplikat:", df.shape)

df = df.drop_duplicates(subset="content").reset_index(drop=True)
print("Setelah drop duplikat:", df.shape)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 32.3 MB/s eta 0:00:00
Sebelum drop duplikat: (3000, 3)
Setelah drop duplikat: (2846, 3)


In [ ]:
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import emoji
import unicodedata

# --- peta huruf small caps (ᴡᴀʟᴀᴜᴘᴜɴ dsb) -> huruf latin biasa; tidak tertangani oleh NFKD ---
SMALLCAPS_MAP = {
    "ᴀ":"a","ʙ":"b","ᴄ":"c","ᴅ":"d","ᴇ":"e","ꜰ":"f","ɢ":"g","ʜ":"h","ɪ":"i",
    "ᴊ":"j","ᴋ":"k","ʟ":"l","ᴍ":"m","ɴ":"n","ᴏ":"o","ᴘ":"p","ǫ":"q","ʀ":"r",
    "ѕ":"s","ᴛ":"t","ᴜ":"u","ᴠ":"v","ᴡ":"w","x":"x","ʏ":"y","ᴢ":"z",
}

# --- kamus emoji -> tag sentimen kasar ---
EMOJI_SENTIMENT = {
    "👍": "emoji_positif", "🥰": "emoji_positif", "😍": "emoji_positif",
    "❤️": "emoji_positif", "💜": "emoji_positif", "😊": "emoji_positif",
    "🤩": "emoji_positif", "😁": "emoji_positif", "👏": "emoji_positif",
    "💯": "emoji_positif", "😘": "emoji_positif", "✨": "emoji_positif",
    "😇": "emoji_positif", "🎉": "emoji_positif", "😄": "emoji_positif",
    "🌟": "emoji_positif",
    "😭": "emoji_negatif", "😡": "emoji_negatif", "🤬": "emoji_negatif",
    "👎": "emoji_negatif", "😤": "emoji_negatif", "🥲": "emoji_negatif",
    "😢": "emoji_negatif", "💢": "emoji_negatif", "🤢": "emoji_negatif",
    "😞": "emoji_negatif", "😔": "emoji_negatif", "💔": "emoji_negatif",
    "😩": "emoji_negatif", "😠": "emoji_negatif", "🙄": "emoji_negatif",
    "👹": "emoji_negatif", "👺": "emoji_negatif",
    "🗿": "emoji_netral", "😹": "emoji_netral", "🤣": "emoji_netral",
    "😂": "emoji_netral", "🤡": "emoji_netral", "💀": "emoji_netral",
    "😅": "emoji_netral", "🙏": "emoji_netral",
}

def extract_emoji_tags(text):
    """Keluarkan semua emoji jadi tag sentimen, kembalikan (teks_tanpa_emoji, list_tag)."""
    tags = []
    for e, tag in EMOJI_SENTIMENT.items():
        count = text.count(e)
        if count:
            tags.extend([tag] * count)
            text = text.replace(e, " ")
    remaining = emoji.emoji_list(text)          # emoji lain yang tidak ada di kamus
    tags.extend(["emoji_lain"] * len(remaining))
    text = emoji.replace_emoji(text, replace=" ")
    return text, tags

# --- kamus normalisasi kata gaul / tidak baku (termasuk istilah khas ML) ---
SLANG_DICT = {
    # variasi typo "moonton" (nama publisher, banyak sekali variannya di data)
    "monton": "moonton", "muntun": "moonton", "montoon": "moonton",
    "montol": "moonton", "montod": "moonton", "montool": "moonton",
    "montoll": "moonton", "monoton": "moonton", "mootoon": "moonton",
    "mooton": "moonton", "muntoon": "moonton",
    # kata ganti & partikel informal
    "gw": "saya", "gue": "saya", "gua": "saya", "aq": "saya", "ak": "saya",
    "lu": "kamu", "lo": "kamu", "km": "kamu",
    "yg": "yang", "dr": "dari", "tp": "tapi", "dgn": "dengan", "sm": "sama",
    "dpt": "dapat", "jd": "jadi", "blm": "belum", "udh": "sudah", "dah": "sudah",
    "kl": "kalau", "klo": "kalau", "kalo": "kalau", "gmn": "bagaimana",
    "knp": "kenapa", "sy": "saya", "bgt": "banget", "bngt": "banget",
    "gk": "tidak", "ga": "tidak", "nggak": "tidak", "engga": "tidak",
    "enggak": "tidak", "gak": "tidak", "kaga": "tidak",
    "jgn": "jangan", "trs": "terus", "utk": "untuk", "krn": "karena",
    "karna": "karena", "bs": "bisa", "biar": "supaya",
    # istilah khas komunitas Mobile Legends
    "ws": "winstreak", "ls": "losestreak", "wr": "winrate",
    "cit": "cheater", "citer": "cheater",
    "mm": "matchmaking", "hp": "handphone",
}

def normalize_slang(text):
    words = text.split()
    return " ".join(SLANG_DICT.get(w, w) for w in words)

def clean_text(text):
    text = re.sub(r"http\S+|www\.\S+", " ", text)      # URL
    text = unicodedata.normalize("NFKD", text)           # ubah huruf gaya (bold script, dsb) -> huruf latin biasa
    text = "".join(SMALLCAPS_MAP.get(ch, ch) for ch in text)  # tangani small caps (ᴡᴀʟᴀᴜᴘᴜɴ dsb) yang tidak tertangani NFKD
    text = re.sub(r"\d+", " ", text)                    # angka
    text = re.sub(r"[^a-z\s_]", " ", text)               # selain huruf latin -> spasi
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)            # huruf berulang berlebih
    text = re.sub(r"\s+", " ", text).strip()
    return text

# --- stopword removal (kata negasi TIDAK dihapus agar makna sentimen tidak berbalik) ---
stopword_factory = StopWordRemoverFactory()
stopwords = set(stopword_factory.get_stop_words())
NEGATION_KEEP = {"tidak", "bukan", "jangan", "kurang", "tanpa", "belum"}
stopwords = stopwords - NEGATION_KEEP

def remove_stopwords(text):
    return " ".join(w for w in text.split() if w not in stopwords)

# --- stemmer ---
stemmer = StemmerFactory().create_stemmer()

def full_pipeline(text, use_stemming=False):
    text = text.lower()
    text, emoji_tags = extract_emoji_tags(text)
    text = clean_text(text)
    text = normalize_slang(text)
    text = remove_stopwords(text)
    if use_stemming:
        text = stemmer.stem(text)
    text = re.sub(r"\s+", " ", text).strip()
    if emoji_tags:
        text = (text + " " + " ".join(emoji_tags)).strip()
    return text

print("Fungsi preprocessing siap dipakai.")

Fungsi preprocessing siap dipakai.


In [ ]:
df["clean_no_stem"] = df["content"].apply(lambda t: full_pipeline(t, use_stemming=False))
df["clean_stem"]    = df["content"].apply(lambda t: full_pipeline(t, use_stemming=True))

df[["content", "clean_no_stem", "clean_stem"]].head(10)

,content,clean_no_stem,clean_stem
0,woy monton bisa buat game gak udah empat belas...,woy moonton buat game tidak udah empat belas k...,woy moonton buat game tidak udah empat belas k...
1,KALAU NGASIH TEAM YG BENER LAHHH,kalau ngasih team bener lahh,kalau ngasih team bener lahh
2,mooton bujang intinya game pp game bujang main...,moonton bujang intinya game pp game bujang mai...,moonton bujang inti game pp game bujang main k...
3,capek juga main selalu kalah saya sebagai solo...,capek main selalu kalah solo rankk tim selalu ...,capek main selalu kalah solo rankk tim selalu ...
4,"Pengen balik ke ml tolong, jangan tambahkan fi...",pengen balik ml jangan tambahkan fitur fitur b...,ken balik ml jangan tambah fitur fitur bikin n...
5,game sistem gj kasih aja gw kalahh terus tim m...,game sistem gj kasih aja kalahh terus tim musu...,game sistem gj kasih aja kalahh terus tim musu...
6,game gateli lose trike tros,game gateli lose trike tros,game gateli lose trike tros
7,makasih moontoon,makasih moontoon,makasih moontoon
8,Sumpah ini moonton kejam dah.Lagi asik main ti...,sumpah moonton kejam asik main tiba malah rest...,sumpah moonton kejam asik main tiba malah rest...
9,bagus keren,bagus keren,bagus keren


In [ ]:
vocab_no_stem = set(" ".join(df["clean_no_stem"]).split())
vocab_stem    = set(" ".join(df["clean_stem"]).split())

avg_len_no_stem = df["clean_no_stem"].apply(lambda t: len(t.split())).mean()
avg_len_stem    = df["clean_stem"].apply(lambda t: len(t.split())).mean()

print("=== Tanpa stemming ===")
print("Ukuran vocab :", len(vocab_no_stem))
print("Rata-rata token/ulasan:", round(avg_len_no_stem, 2))

print("\n=== Dengan stemming ===")
print("Ukuran vocab :", len(vocab_stem))
print("Rata-rata token/ulasan:", round(avg_len_stem, 2))

print("\nSelisih vocab (tereduksi oleh stemming):", len(vocab_no_stem) - len(vocab_stem))

=== Tanpa stemming ===
Ukuran vocab : 4899
Rata-rata token/ulasan: 14.36

=== Dengan stemming ===
Ukuran vocab : 4079
Rata-rata token/ulasan: 14.36

Selisih vocab (tereduksi oleh stemming): 820


In [ ]:
kosong_no_stem = (df["clean_no_stem"].str.strip() == "").sum()
kosong_stem    = (df["clean_stem"].str.strip() == "").sum()
print("Baris kosong (no stem):", kosong_no_stem)
print("Baris kosong (stem)   :", kosong_stem)

df[df["clean_no_stem"].str.strip() == ""][["content", "score"]]

Baris kosong (no stem): 1
Baris kosong (stem)   : 1


,content,score
342,ok,5


In [ ]:
# drop baris yang hasil cleaning-nya kosong (hanya berlaku untuk clean_no_stem;
# baris yang sama juga kosong di clean_stem karena stemming tidak bikin teks jadi ada isi dari kosong)
df = df[df["clean_no_stem"].str.strip() != ""].reset_index(drop=True)
print("Jumlah baris setelah drop baris kosong:", len(df))

Jumlah baris setelah drop baris kosong: 2845


In [ ]:
df.to_csv("/content/ml_sample_preprocessed.csv", index=False)
print("Tersimpan: /content/ml_sample_preprocessed.csv")
print("Kolom:", list(df.columns))
print("Jumlah baris final:", len(df))

Tersimpan: /content/ml_sample_preprocessed.csv
Kolom: ['content', 'score', 'at', 'clean_no_stem', 'clean_stem']
Jumlah baris final: 2845
